# Modeling Financial Information over Time

The first object describes uncertainty, and the nested sets describe information revealed through time:

$$
(\Omega,\mathcal F,\mathbb P),\qquad
\mathcal F_0\subseteq\mathcal F_1\subseteq\cdots\subseteq\mathcal F_T,
\qquad R_t\in\mathcal F_t.
$$

Here, $\Omega$ is the set of possible outcomes; $\mathcal F$ is the set of events that may be assigned probabilities. $\mathbb P$ assigns probabilities; $\mathcal F_t$ is the information known by time $t$; $T$ is the final time. $R_t$ is the return observed at time $t$; $R_t\in\mathcal F_t$ says that this return is known at that time.

## A Financial Filtration

A return is written as a predictable part plus a scaled surprise:

$$
R_t=\mu_t\Delta t+\sigma_t\sqrt{\Delta t}\,\varepsilon_t,
\qquad
\mathbb E[\varepsilon_t\mid\mathcal F_{t-1}]=0,
\qquad
\mathbb E[R_t\mid\mathcal F_{t-1}]=\mu_t\Delta t.
$$

Here, $R_t$ is the return; $\mu_t$ is its conditional drift rate; $\sigma_t$ is its conditional volatility. $\Delta t$ is the length of the time interval; $\varepsilon_t$ is a zero-mean shock; $\mathcal F_{t-1}$ is prior information. $\mathbb E[\,\cdot\mid\mathcal F_{t-1}]$ is an expectation using that prior information.

In [1]:
import re as _table_number_re
from IPython.display import HTML as _TableHTML, display as _raw_display


def compact_float_format(number_format=None):
    def format_value(value):
        if number_format is None:
            rendered = str(value)
        elif callable(number_format):
            rendered = str(number_format(value))
        else:
            rendered = number_format % value
        return compact_numeric_token(rendered)
    return format_value


def compact_numeric_token(rendered):
    if not _table_number_re.fullmatch(r"[+-]?(?:[0-9]+(?:\.[0-9]*)?|\.[0-9]+)(?:[eE][+-]?[0-9]+)?", rendered):
        return rendered
    value = float(rendered)
    if value != 0.0 and abs(value) < 5e-13:
        return "0"
    if ("." in rendered or "e" in rendered.lower()) and value.is_integer():
        return str(int(value))
    return rendered


def compact_numeric_text(text):
    pattern = r"(?<![0-9.eE])([+-]?(?:[0-9]+(?:\.[0-9]*)?|\.[0-9]+)(?:[eE][+-]?[0-9]+)?)(?![0-9.eE])"
    return _table_number_re.sub(pattern, lambda match: compact_numeric_token(match.group(0)), text)


def compact_table_latex(latex):
    return compact_numeric_text(latex)


def _compact_table_html(value):
    frame = value.to_frame() if isinstance(value, pd.Series) else value
    html = frame._repr_html_()
    return '<div style="display:flex;justify-content:center;">' + compact_numeric_text(html) + "</div>"


def display(value):
    if isinstance(value, (pd.DataFrame, pd.Series)):
        return _raw_display(_TableHTML(_compact_table_html(value)))
    return _raw_display(value)

from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


ROOT = Path.cwd().parents[1]
import RiskLabAI

assert RiskLabAI.__version__ == "3.2.0"

from RiskLabAI.data.synthetic_data.drift_burst_hypothesis import drift_volatility_burst

FIGURES = ROOT / "outputs" / "figures" / "01_mathematical_language"
TABLES = ROOT / "outputs" / "tables" / "01_mathematical_language"
FIGURE_DATA = ROOT / "outputs" / "figure_data" / "01_mathematical_language"
FIGURES.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)
FIGURE_DATA.mkdir(parents=True, exist_ok=True)

palette = {
    "background": "#FFFFFF",
    "axes": "#FFFFFF",
    "text": "#1B365D",
    "cyan": "#0072B2",
    "amber": "#C76D00",
    "green": "#00866A",
    "rose": "#B83B3B",
    "grid": "#D9DEE5",
}
plt.rcParams.update({
    "font.family": "DejaVu Serif",
    "figure.facecolor": palette["background"],
    "axes.facecolor": palette["axes"],
    "axes.edgecolor": palette["text"],
    "axes.labelcolor": palette["text"],
    "axes.titlecolor": palette["text"],
    "xtick.color": palette["text"],
    "ytick.color": palette["text"],
    "text.color": palette["text"],
    "grid.color": palette["grid"],
    "grid.linestyle": ":",
    "legend.facecolor": palette["axes"],
    "legend.edgecolor": palette["grid"],
    "savefig.facecolor": palette["background"],
})

## RiskLabAI Experiment

In [2]:
observation_count = 401
time = np.linspace(0.0, 1.0, observation_count)
drift, volatility = drift_volatility_burst(
    bubble_length=observation_count,
    a_before=0.025,
    a_after=-0.018,
    b_before=0.12,
    b_after=0.16,
    alpha=0.25,
    beta=0.18,
    explosion_filter_width=0.06,
)
time_step = 1.0 / 252.0
random_generator = np.random.default_rng(20260925)
innovation = random_generator.standard_normal(observation_count)
conditional_mean = drift * time_step
return_path = conditional_mean + volatility * np.sqrt(time_step) * innovation
price_path = 100.0 * np.exp(np.cumsum(return_path))
standardized_innovation = (return_path - conditional_mean) / (volatility * np.sqrt(time_step))
information_count = np.arange(1, observation_count + 1)

selected = np.array([0, 100, 200, 300, 400])
information_table = pd.DataFrame({
    "Observation": selected + 1,
    "Normalized time": time[selected],
    "Observed returns in information set": information_count[selected],
    "Price": price_path[selected],
    "Conditional drift": drift[selected],
    "Conditional volatility": volatility[selected],
})
information_table.to_csv(TABLES / "information_checkpoints.csv", index=False)
information_figure_data = pd.DataFrame({
    "Normalized time": time,
    "Price": price_path,
    "Conditional drift": drift,
    "Conditional volatility": volatility,
    "Information checkpoint": np.isin(np.arange(observation_count), selected),
})
information_figure_data.to_csv(FIGURE_DATA / "information_filtration.csv", index=False)
information_latex = compact_table_latex(information_table.to_latex(
    index=False,
    float_format=compact_float_format("%.4f"),
    caption="Information available at selected points of the simulated price path.",
    label="tab:information-checkpoints",
    position="tbp",
))
information_latex = information_latex.replace(
    r"\begin{tabular}",
    r"\begin{adjustbox}{max width=\textwidth}\begin{tabular}",
    1,
).replace(r"\end{tabular}", r"\end{tabular}\end{adjustbox}", 1)
(TABLES / "information_checkpoints.tex").write_text(information_latex, encoding="utf-8")
display(information_table.round(4))

,Observation,Normalized time,Observed returns in information set,Price,Conditional drift,Conditional volatility
0,1,0,1,100.3855,0.0297,0.1359
1,101,0.25,101,105.3998,0.0354,0.1540
2,201,0.50,201,120.6279,0,0.1991
3,301,0.75,301,161.4873,-0.0255,0.2053
4,401,1,401,155.9880,-0.0214,0.1813


In [3]:
figure, axes = plt.subplots(3, 1, figsize=(10.5, 8.6), sharex=True, constrained_layout=True)
axes[0].plot(time, price_path, color=palette["cyan"], linewidth=2.2)
axes[0].set_ylabel("Price")
axes[0].set_title("Information Accumulates Along the Observed Price Path")
axes[1].plot(time, drift, color=palette["amber"], linewidth=2.0)
axes[1].axhline(0.0, color=palette["text"], linewidth=0.8)
axes[1].set_ylabel("Drift")
axes[2].plot(time, volatility, color=palette["green"], linewidth=2.0)
axes[2].set_ylabel("Volatility")
axes[2].set_xlabel("Normalized time")
for axis in axes:
    for location in time[selected]:
        axis.axvline(location, color=palette["rose"], alpha=0.22, linewidth=0.8)
figure.savefig(FIGURES / "information_filtration.png", bbox_inches="tight")
figure.savefig(FIGURES / "information_filtration.pdf", bbox_inches="tight")
display(figure)
plt.close(figure)

<Figure size 1050x860 with 3 Axes>

## Innovations

The innovation is the part of the return that could not be predicted from earlier information:

$$
U_t=R_t-\mathbb E[R_t\mid\mathcal F_{t-1}],
\qquad
\mathbb E[U_t\mid\mathcal F_{t-1}]=0.
$$

Here, $U_t$ is the innovation; $R_t$ is the observed return; $\mathcal F_{t-1}$ is information known one step earlier. $\mathbb E[R_t\mid\mathcal F_{t-1}]$ is the predicted return, and the second equality states that the remaining surprise has conditional mean zero.

In [4]:
density_grid = np.linspace(-3.8, 3.8, 300)
normal_density = np.exp(-0.5 * density_grid**2) / np.sqrt(2.0 * np.pi)
running_mean = np.cumsum(standardized_innovation) / information_count
innovation_series = pd.DataFrame({
    "Observed returns": information_count,
    "Standardized innovation": standardized_innovation,
    "Running mean": running_mean,
})
innovation_density = pd.DataFrame({
    "Standardized innovation": density_grid,
    "Standard normal density": normal_density,
})
innovation_series.to_csv(FIGURE_DATA / "innovation_series.csv", index=False)
innovation_density.to_csv(FIGURE_DATA / "innovation_density.csv", index=False)

figure, axes = plt.subplots(1, 2, figsize=(10.5, 4.3), constrained_layout=True)
axes[0].hist(standardized_innovation, bins=28, density=True, color=palette["cyan"], alpha=0.78)
axes[0].plot(density_grid, normal_density, color=palette["amber"], linewidth=2.1, label="Standard normal density")
axes[0].set_xlabel("Standardized innovation")
axes[0].set_ylabel("Density")
axes[0].set_title("Innovation Distribution")
axes[0].legend()
axes[1].plot(information_count, running_mean, color=palette["green"], linewidth=1.8)
axes[1].axhline(0.0, color=palette["amber"], linewidth=1.0)
axes[1].set_xlabel("Observed returns")
axes[1].set_ylabel("Running mean")
axes[1].set_title("Conditional-Mean Diagnostic")
figure.savefig(FIGURES / "innovation_diagnostics.png", bbox_inches="tight")
figure.savefig(FIGURES / "innovation_diagnostics.pdf", bbox_inches="tight")
display(figure)
plt.close(figure)

<Figure size 1050x430 with 2 Axes>

## Tests

In [5]:
midpoint = observation_count // 2
repeated_generator = np.random.default_rng(20260925)
repeated_innovation = repeated_generator.standard_normal(observation_count)
assert drift.shape == (observation_count,)
assert volatility.shape == (observation_count,)
assert drift[midpoint] == 0.0
assert volatility[midpoint] == volatility[midpoint - 1]
assert np.all(volatility > 0.0)
assert np.all(np.isfinite(price_path))
assert np.all(price_path > 0.0)
assert np.array_equal(innovation, repeated_innovation)
assert abs(standardized_innovation.mean()) < 0.15
assert len(information_table) == len(selected)